# KARTA — Demo segmentasi sampah (foto, kamera, video konveyor)

**Tujuan notebook ini:** memeriksa alur KARTA dari ujung ke ujung *sebelum* ada foto lapangan dan anotasi:
foto/kamera/video → segmentasi fraksi (kotak + warna) → persen luas tiap fraksi → kadar air + interval → LHV → rute.

**Penting, baca dulu**
- Mode awal `zero-shot` memakai CLIPSeg yang **tidak dilatih dengan sampah Putri Cempo**. Ia hanya dipandu teks ("plastic bag", "food waste", ...). Hasilnya **bukan kinerja KARTA**. Gunanya: melihat bentuk keluaran dan menguji alur kamera/video.
- Kelas yang mirip (organik basah vs kertas basah vs tekstil) sering tertukar di mode ini. Itu wajar, dan itulah alasan kita nanti melatih SegFormer dari foto yang dianotasi.
- Kadar air di sini memakai **model fisika dengan koefisien literatur** (belum dikalibrasi uji oven). Angkanya ilustrasi, bukan ukuran.

**Cara pakai:** Runtime → Change runtime type → **T4 GPU** (lebih cepat, CPU juga bisa tapi lambat) → jalankan sel dari atas ke bawah.

## 1. Unggah dan ekstrak paket prototipe

In [ ]:
# Pilih salah satu: (A) unggah zip langsung, atau (B) ambil dari Google Drive.
import os, zipfile, glob
from google.colab import files

CARA = "A"   # ganti "B" kalau zip ada di Google Drive
if CARA == "A":
    up = files.upload()                       # pilih prototipe_v3.zip
    ZIP = list(up.keys())[0]
else:
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP = "/content/drive/MyDrive/prototipe_v3.zip"   # ubah sesuai lokasi

os.makedirs("/content/karta_pkg", exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content/karta_pkg")
akar = os.path.dirname(glob.glob("/content/karta_pkg/**/karta/konstanta.py", recursive=True)[0]).rsplit("/karta", 1)[0]
print("Akar proyek:", akar)

## 2. Pasang pustaka dan muat model

In [ ]:
!pip -q install transformers opencv-python-headless scipy scikit-learn lightgbm pandas matplotlib
import sys, cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, akar)
from karta.visi import muat_model
from karta.kadar_air import ModelKadarAir
from karta.tampil import gambar_hasil, bandingkan
from karta.aliran import proses_video, proses_deret_foto, PemantauAliran
from karta.konstanta import FRAKSI

# MODE: "zero-shot" = tanpa pelatihan (sekarang). Nanti, setelah ada hasil fine-tune: "segformer" + PATH bobot.
MODE, PATH = "zero-shot", None
model_visi = muat_model(MODE, PATH)
model_air = ModelKadarAir("fisika")      # nanti diganti "koreksi_linear" setelah ada data uji oven
print("Model siap:", MODE)

## 3. Satu foto (unggah dari laptop atau ambil dari Google)
Simpan foto sampah dari Google/foto sendiri, lalu unggah. Hasil: **asli | segmentasi + kotak | peta keraguan model**.

In [ ]:
from google.colab import files
from google.colab.patches import cv2_imshow

def baca_unggahan():
    up = files.upload()
    return {n: cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_COLOR) for n, b in up.items()}

def tampilkan(bgr, judul=""):
    hv = model_visi(bgr)
    cv2_imshow(bandingkan(bgr, hv))
    print({f: f"{hv.proporsi[f]*100:.0f}%" for f in FRAKSI}, "| keyakinan rata-rata:", round(hv.keyakinan, 2))
    return hv

foto = baca_unggahan()
for nama, im in foto.items():
    print(nama); hv = tampilkan(im)

### 3b. Foto dari alamat web (URL gambar)
Klik kanan foto di Google → *Copy image address*, tempel di bawah.

In [ ]:
import urllib.request
URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b0/Garbage_dump.jpg/640px-Garbage_dump.jpg"  # ganti
req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
data = urllib.request.urlopen(req, timeout=30).read()
im = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
hv = tampilkan(im)

## 4. Atur sensitivitas (kalau hasil terlalu kosong / terlalu penuh)
- `ambang` naik → lebih banyak area dianggap *latar* (lebih selektif). Turun → lebih banyak area diberi kelas.
- `grid` 2 = foto dipotong 2×2 lalu digabung (batas lebih halus, 4× lebih lambat). Coba 1 untuk cepat.
- Prompt teks bisa diganti sesuai isi foto Anda (bahasa Inggris lebih bagus untuk CLIPSeg).

In [ ]:
model_visi.ambang = 0.30
model_visi.grid = 2
# Contoh mengganti prompt:
# from karta.zero_shot import PROMPT_ZS
# PROMPT_ZS["organik_basah"] = ["food scraps", "rotten fruit", "banana peel"]
# model_visi = muat_model("zero-shot")
hv = tampilkan(im)

## 5. Kamera laptop di Colab: satu jepretan
Browser akan meminta izin kamera. Tombol "Ambil foto" muncul di bawah sel.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def ambil_foto(kualitas=0.9):
    js = Javascript('''
    async function ambil(q) {
      const div = document.createElement('div');
      const btn = document.createElement('button'); btn.textContent = 'Ambil foto'; div.appendChild(btn);
      const video = document.createElement('video'); video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: 'environment'}});
      document.body.appendChild(div); div.appendChild(video);
      video.srcObject = stream; await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((r) => btn.onclick = r);
      const c = document.createElement('canvas'); c.width = video.videoWidth; c.height = video.videoHeight;
      c.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop(); div.remove();
      return c.toDataURL('image/jpeg', q);
    }''')
    display(js)
    d = eval_js('ambil(%f)' % kualitas)
    return cv2.imdecode(np.frombuffer(b64decode(d.split(',')[1]), np.uint8), cv2.IMREAD_COLOR)

im = ambil_foto()
hv = tampilkan(im)

## 6. Simulasi konveyor dengan kamera laptop (foto berurutan)
Arahkan kamera ke tumpukan benda yang digeser pelan di depan kamera (atau layar HP yang memutar video).
Kamera memotret otomatis tiap beberapa detik; tiap jendela waktu menghasilkan **satu keputusan rute**.

In [ ]:
import time
from IPython.display import display, Javascript
from google.colab.output import eval_js

def mulai_kamera():
    display(Javascript('''
    window.__kam = null;
    (async () => {
      const v = document.createElement('video'); v.style.width = '320px'; v.autoplay = true; v.playsInline = true;
      const s = await navigator.mediaDevices.getUserMedia({video: true});
      v.srcObject = s; document.body.appendChild(v); await v.play(); window.__kam = {v, s};
      google.colab.output.setIframeHeight(280, true);
    })();'''))
    time.sleep(4)

def jepret():
    d = eval_js('''(() => { const k = window.__kam; const c = document.createElement('canvas');
        c.width = k.v.videoWidth; c.height = k.v.videoHeight; c.getContext('2d').drawImage(k.v, 0, 0);
        return c.toDataURL('image/jpeg', 0.9); })()''')
    return cv2.imdecode(np.frombuffer(b64decode(d.split(',')[1]), np.uint8), cv2.IMREAD_COLOR)

def matikan_kamera():
    eval_js("window.__kam && window.__kam.s.getTracks().forEach(t => t.stop())")

JUMLAH, JEDA, JENDELA = 12, 2.0, 8.0     # 12 foto, tiap 2 detik, 1 keputusan per 8 detik
mulai_kamera()
frames = []
for i in range(JUMLAH):
    frames.append(jepret()); time.sleep(JEDA)
matikan_kamera()
gambar, tabel_waktu, tabel_keputusan = proses_deret_foto(frames, JEDA, model_visi, model_air, jendela_detik=JENDELA)
for g in gambar[::3]:
    cv2_imshow(g)
tabel_keputusan

## 7. Video (rekaman konveyor, atau video sampah dari internet)
Unggah file video (mp4/mov, sebaiknya < 1 menit). Keluaran: video bertanda kotak + tabel keputusan per jendela waktu.
Segmentasi dijalankan tiap `TIAP_DETIK`, hasil terakhir digambar di semua frame di antaranya.

In [ ]:
from google.colab import files
up = files.upload()
VID = list(up.keys())[0]
TIAP_DETIK, JENDELA = 1.0, 8.0
tabel_waktu, tabel_keputusan, hasil_mp4 = proses_video(
    VID, "/content/hasil_mentah.mp4", model_visi, model_air,
    tiap_detik=TIAP_DETIK, jendela_detik=JENDELA, maks_detik=60, lebar_maks=960)
# Ubah ke H.264 supaya bisa diputar di browser
!ffmpeg -y -loglevel error -i /content/hasil_mentah.mp4 -vcodec libx264 -pix_fmt yuv420p /content/hasil_segmentasi.mp4
from IPython.display import HTML
from base64 import b64encode
b64 = b64encode(open("/content/hasil_segmentasi.mp4", "rb").read()).decode()
display(HTML(f'<video width=800 controls src="data:video/mp4;base64,{b64}"></video>'))
tabel_keputusan[["t_awal","t_akhir","kadar_air","kadar_air_bawah","kadar_air_atas","lhv_mj_kg","rute","nama_rute","perlu_cek_manual"]]

## 8. Grafik komposisi terhadap waktu dan simpan hasil

In [ ]:
ax = tabel_waktu.set_index("t")[FRAKSI].plot.area(figsize=(10, 3.5), alpha=.85)
ax.set_xlabel("detik"); ax.set_ylabel("porsi luas"); ax.set_title("Komposisi fraksi sepanjang video"); plt.show()
tabel_waktu.to_csv("/content/komposisi_per_waktu.csv", index=False)
tabel_keputusan.to_csv("/content/keputusan_per_jendela.csv", index=False)
files.download("/content/komposisi_per_waktu.csv"); files.download("/content/keputusan_per_jendela.csv")

## 9. Cara membaca hasil dan langkah berikutnya
| Yang Anda lihat | Artinya |
|---|---|
| Kotak berwarna + label | Area terhubung satu kelas (bukan deteksi objek per benda). Persen = porsi luas gambar |
| Panel kanan | Komposisi luas tiap fraksi pada frame itu |
| Peta keraguan | Terang = model ragu (probabilitas kelas bersaing). Dipakai untuk menandai "perlu cek manual" |
| `kadar_air` + rentang | Perkiraan fisika dari komposisi. **Belum dikalibrasi**, jangan dikutip sebagai kinerja |
| `rute` | U umpan langsung, B bio-drying + serbuk kayu, M maggot BSF, R residu/tunda |

**Setelah ada data:** (1) kumpulkan foto sesuai protokol, (2) anotasi 100–300 foto di CVAT dengan bantuan SAM2,
(3) latih `latih/05_latih_segformer.py` di Colab, (4) ubah sel 2 menjadi `MODE, PATH = "segformer", "/content/segformer_karta"`.
Semua sel lain tetap sama.